# 54. 설정 동결 후 새32질문·3회 확인
52번의 네 검색 방식과 53번의 근거 보완 설정을 정한 뒤 질문을 작성·동결한다. 같은106개 문서에 대한 신규 질문 확인이며 새로운 PDF의 독립 검증은 아니다. 운영 코드·청킹·원본 DB는 변경하지 않는다. 모든 방식의 예정672조건을 유지하며 반복 간 답변은 공유하지 않는다. 최초 토큰/브라우저 시간은 미측정이고, 배치 단계 합계를 실제 온라인 종단 시간으로 부르지 않는다.

공유 Python 모듈은 재개 시 입력·조회·원본 보존·요청 신원·8달러 상한 검사를 동일하게 적용한다. 질문, 실행 순서와 평가는 이 노트북에 둔다. HTML 보고서는 결과 확인 뒤 직접 작성한다.

질문 해결(`question_verdict`)과 부가 원문 주석 포함률(`annotation_coverage`)을 분리한다. 두 노트북에서 같은 판정 검사를 재사용·검증하기 위해 작은 오프라인 helper를 사용한다. 질문별 필수 답변·중요 적용조건·부가 주석은 유료 답변 생성 전에 동결한다.

In [1]:
from pathlib import Path
import importlib.util,sys,json,os,time,copy,resource
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
spec=importlib.util.spec_from_file_location('confirm_modes54',ROOT/'notebooks/scripts/52_retrieval_modes.py')
M=importlib.util.module_from_spec(spec);sys.modules[spec.name]=M;spec.loader.exec_module(M)
X=M.X;P=M.P;Q=M.Q
X.load('confirm_recovery54','51_budget_recovery.py').install(X)
C=X.load('confirm_predicate54','52_query_predicate_correction.py');C.install(X)
PATH=P.F.STAGES[54];PATH.mkdir(exist_ok=True);P.F.initialize(54)
assert P.F.read_json(P.F.STAGES[53]/'completion_status.json')['stage54_allowed']
selection=P.F.read_json(PATH/'frozen_configuration.json')
prior52=P.F.read_json(P.F.STAGES[52]/'evaluation_summary.json');prior53=P.F.read_json(P.F.STAGES[53]/'evaluation_summary.json')
assert selection['upstream52_sha256']==P.sha(prior52) and selection['upstream53_sha256']==P.sha(prior53)
assert selection['parallel_fusion']=='BGE'  # LLM selection needs its own expanded physical-call review before freeze.
selected_AB={r['profile']:r['selected'] for r in prior53['decisions']}
assert selection['SEQ_supplement_by_profile']==selected_AB
queries=P.F.read_json(PATH/'questions.json')['queries'];annotations=P.F.read_json(PATH/'source_annotations.json')
assert len(queries)==32 and len({q['query_id'] for q in queries})==32 and len({q['question'] for q in queries})==32
from collections import Counter
assert Counter(q['subgroup'] for q in queries)==dict(benefit=8,compound_detail=8,issuer_product=8,numeric=4,negative=4)
previous_questions=P.F.read_json(P.F.STAGES[50]/'questions.json')['queries']
assert not ({q['question'] for q in queries}&{q['question'] for q in previous_questions})
assert {r['query_id'] for r in annotations['queries']}=={q['query_id'] for q in queries}
gold_by_query={r['query_id']:r['gold_cards'] for r in annotations['queries']}
import fitz
source_inputs={'data/raw/'+d['document_id']+'.pdf':d['source_pdf_sha256'] for d in Q.documents()}
source_page_counts={d['document_id']:len(fitz.open(ROOT/('data/raw/'+d['document_id']+'.pdf'))) for d in Q.documents()}
source_inputs.update({str((P.DATA/'30_structured_benefit_db'/name).relative_to(ROOT)):P.sha((P.DATA/'30_structured_benefit_db'/name).read_bytes()) for name in ['structured.sqlite','documents.jsonl','facts.jsonl']})
for name,wanted in source_inputs.items():assert P.sha((ROOT/name).read_bytes())==wanted
# Shared offline evaluator; bind its rules and rubrics before paid outputs.
E=X.load('question_evaluation_54','53_question_evaluation.py')
rubric_path=PATH/'question_rubrics.json'
rubrics=P.F.read_json(rubric_path);rubric_by_query={r['query_id']:r for r in rubrics['records']}
assert set(rubric_by_query)=={q['query_id'] for q in queries}
for q in queries:assert rubric_by_query[q['query_id']]['question']==q['question']
evaluation_inputs={str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in [rubric_path,P.F.STAGES[52]/'evaluation_interpretation_supplement.json']}
evaluation_sources={**P.F.read_json(P.F.STAGES[52]/'execution_manifest.json')['source_sha256'],'notebooks/scripts/53_question_evaluation.py':P.sha((ROOT/'notebooks/scripts/53_question_evaluation.py').read_bytes())}
identities=[f"{q['query_id']}:S_ONLY:r{r}" for r in [1,2,3] for q in queries]+[f"{q['query_id']}:{pf}:{mode}:r{r}" for r in [1,2,3] for q in queries for pf in X.PROFILES for mode in ['U','SEQ','PAR']]
parser_ids=[f"{q['query_id']}:parser:r{r}" for r in [1,2,3] for q in queries]
manifest=X.make_manifest(54,ROOT/'notebooks/54_new_question_confirmation.ipynb',queries,dict(parser=parser_ids,answer=identities),maximum_parser=96,maximum_answer=1344,maximum_embedding=1,stage_cap='4',extra=dict(
 source_sha256=evaluation_sources,question_rubrics_sha256=P.sha(rubrics),primary_metric='question_verdict',secondary_metric='annotation_coverage',split='confirmatory_same_corpus',logical_answer_conditions=672,repeats=3,
 source_input_sha256=source_inputs,source_page_counts=source_page_counts,corpus_documents_sha256=P.sha(Q.documents()),selection_sha256=P.sha(selection),annotations_sha256=P.sha(annotations),
 data_sha256={**evaluation_inputs,**P.F.read_json(P.F.STAGES[52]/'execution_manifest.json')['data_sha256'],**{str(p.relative_to(ROOT)):P.sha(p.read_bytes()) for p in [PATH/'questions.json',PATH/'source_annotations.json',PATH/'frozen_configuration.json',P.F.STAGES[52]/'evaluation_summary.json',P.F.STAGES[53]/'evaluation_summary.json']}},
 decision='net question gain >=1 vs U; no subgroup decline/new important errors/condition loss/scope errors; otherwise no promotion',
 complete_all_repeats_despite_answer_failures=True,operating_changes=False))
ALLOW_NETWORK=os.getenv('PAID_RETEST_ALLOW_NETWORK')=='1'
print('새32질문×3회. 추출96회·답변672조건. 단계상한4달러/누적상한8달러.')

def require_reviewed_query_diff():
 entry_path=PATH/'paid_answer_entry.json'
 if not entry_path.exists():raise RuntimeError('신규 조건 차분 검토가 없어 검색/답변 API를 시작할 수 없다.')
 entry=P.F.read_json(entry_path);probe_file=PATH/'predicate_difference_probe.json';review_file=PATH/'predicate_difference_review.json'
 assert entry['manifest_sha256']==P.sha(manifest)
 assert entry['source_probe_sha256']==P.sha(probe_file.read_bytes()) and entry['review_sha256']==P.sha(P.F.read_json(review_file))
 assert P.F.read_json(probe_file)['parser_results_sha256']==P.sha(P.F.read_json(PATH/'parser_results.json'))
 X.guard_manifest(manifest)
 return entry


새32질문×3회. 추출96회·답변672조건. 단계상한4달러/누적상한8달러.


In [2]:
plans=[]
for repeat in [1,2,3]:
 for q in queries:
  C.guard_installed(X)
  record=X.extract(54,q,repeat,allow_network=ALLOW_NETWORK);plans.append(record)
  P.F.write_json(PATH/'parser_progress.json',dict(completed=len(plans),expected=96,last_identity=record['parser']['identity'],ledger=X.ledger()))
  print('추출',record['parser']['identity'],record['parser']['status'],record['routing']['reason'],flush=True)
X.frozen(PATH/'parser_results.json',dict(records=plans,manifest_sha256=P.sha(manifest)))
by_query={(r['parser']['query_id'],r['parser']['repeat']):r for r in plans}


추출 B01:parser:r1 ok no_results
추출 B02:parser:r1 ok matched
추출 B03:parser:r1 ok no_results
추출 B04:parser:r1 ok no_results
추출 B05:parser:r1 ok matched
추출 B06:parser:r1 ok no_results
추출 B07:parser:r1 ok matched
추출 B08:parser:r1 ok no_results
추출 C01:parser:r1 ok no_results
추출 C02:parser:r1 ok no_results
추출 C03:parser:r1 ok no_results
추출 C04:parser:r1 ok no_results
추출 C05:parser:r1 ok no_results
추출 C06:parser:r1 ok no_results
추출 C07:parser:r1 ok no_results
추출 C08:parser:r1 ok no_results
추출 I01:parser:r1 ok no_results
추출 I02:parser:r1 ok matched
추출 I03:parser:r1 ok no_results
추출 I04:parser:r1 ok no_results
추출 I05:parser:r1 ok no_results
추출 I06:parser:r1 ok no_results
추출 I07:parser:r1 ok matched
추출 I08:parser:r1 ok no_results
추출 N01:parser:r1 ok no_results
추출 N02:parser:r1 ok matched
추출 N03:parser:r1 ok matched
추출 N04:parser:r1 ok no_results
추출 X01:parser:r1 ok no_results
추출 X02:parser:r1 ok no_results
추출 X03:parser:r1 ok no_results
추출 X04:parser:r1 ok no_results
추출 B01:parser:r2 ok no_result

In [ ]:
# 신규 조건의 예외 판정 차분을 유료 답변 전에 다시 검토한다.
original_Q=X.load('confirm_original_query54','50_query_contract.py')
probe_path=PATH/'predicate_difference_probe.json'
if not probe_path.exists():
 requirements={}
 for record in plans:
  value=record['parser']['plan']
  if value is not None:
   for group in value['groups']:
    for req in group:requirements[P.F.canonical(req)]=req
 changes=[]
 for key,req in requirements.items():
  for fid,f in Q.facts_by_id().items():
   before=original_Q.evaluate_fact(f,req);after=Q.evaluate_fact(f,req)
   if before!=after:
    assert not before and after
    changes.append(dict(fact_id=fid,card_key=f['document_id'],requirement=req,before=before,after=after,fields=f['fields'],source_pdf_sha256=f['source_pdf_sha256'],field_evidence=f['field_evidence']))
 X.frozen(probe_path,dict(parser_results_sha256=P.sha(P.F.read_json(PATH/'parser_results.json')),requirements=len(requirements),changed_pairs=changes))
probe=P.F.read_json(probe_path)
assert probe['parser_results_sha256']==P.sha(P.F.read_json(PATH/'parser_results.json'))
review_path=PATH/'predicate_difference_review.json'
if not review_path.exists():raise RuntimeError('신규 조건의 전체 판정 차분 원문 검토 대기. 임베딩·검색·답변 API는 시작하지 않는다.')
review=P.F.read_json(review_path)
assert review['source_probe_sha256']==P.sha(probe_path.read_bytes())
expected={(p['fact_id'],P.sha(p['requirement'])) for p in probe['changed_pairs']}
reviewed={(r['fact_id'],r['requirement_sha256']) for r in review['records']}
owner={(p['fact_id'],P.sha(p['requirement'])):(p['card_key'],p['source_pdf_sha256']) for p in probe['changed_pairs']}
assert expected==reviewed and len(review['records'])==len(expected)
assert all(r['source_confirmed_match'] and r['source_support'] and r['reason'] for r in review['records'])
assert review['uncertain_or_entire_requested_target_exclusion_pairs']==0
for r in review['records']:
 expected_card,expected_pdf=owner[r['fact_id'],r['requirement_sha256']]
 assert r['card_key']==expected_card and Q.facts_by_id()[r['fact_id']]['document_id']==expected_card
 assert Q.facts_by_id()[r['fact_id']]['source_pdf_sha256']==expected_pdf
 for support in r['source_support']:
  key=r['card_key'];name='data/raw/'+key+'.pdf'
  assert support['path']==name and support['sha256']==source_inputs[name]==expected_pdf and 1<=support['page']<=source_page_counts[key]
X.frozen(PATH/'paid_answer_entry.json',dict(manifest_sha256=P.sha(manifest),source_probe_sha256=P.sha(probe_path.read_bytes()),review_sha256=P.sha(review)))
print('신규 조건의 전체 판정 차분 검토 통과. 다음 검색·답변 단계 진입 가능.')


AssertionError: 

In [ ]:
require_reviewed_query_diff()
class BudgetRuntime(X.Runtime):
 def embeddings(self,texts,*,allow_network=False):
  X.guard_manifest(self.manifest)
  start=time.perf_counter();target=self.path/'api_cache/embeddings'
  keys=[P.sha(dict(text=t,model='text-embedding-3-small',dimension=1536)) for t in texts]
  current={}
  batches=[P.F.read_json(p) for p in (self.path/'api_cache').glob('embedding_batch_*.json')]
  for text,key in zip(texts,keys,strict=True):
   path=target/(key+'.npy')
   if not path.exists():continue
   vector=self.np.load(path,allow_pickle=False)
   proof=None
   for batch in batches:
    request=batch.get('request',{})
    if text not in request.get('input',[]):continue
    assert batch['status']=='ok' and request['model']=='text-embedding-3-small' and request['dimensions']==1536
    item=next(r for r in batch['raw_response']['data'] if r['index']==request['input'].index(text))
    proof=self.np.asarray(item['embedding'],dtype=self.np.float32)
   if proof is None or not self.np.array_equal(vector,proof):raise RuntimeError('Current question embedding lacks unchanged raw provider proof')
   current[text]=vector
  remaining=list(dict.fromkeys(t for t in texts if t not in current))
  original_ledger=P.ledger
  try:
   P.ledger=X.budget_upper_ledger
   if remaining:current.update(zip(remaining,super().embeddings(remaining,allow_network=allow_network),strict=True))
  finally:P.ledger=original_ledger
  vectors=self.np.asarray([current[t] for t in texts],dtype=self.np.float32)
  assert vectors.shape==(len(texts),1536) and self.np.isfinite(vectors).all()
  self.last_embedding_seconds=time.perf_counter()-start
  X.frozen(self.path/'question_embedding_contract.json',dict(texts_sha256=P.sha(texts),model='text-embedding-3-small',dimensions=1536,matrix_sha256=P.sha(vectors.tobytes())))
  return vectors
runtime=BudgetRuntime(54)
started=time.perf_counter();cpu_started=time.process_time()
vectors=runtime.embeddings([q['question'] for q in queries],allow_network=ALLOW_NETWORK)
qvectors=dict(zip([q['query_id'] for q in queries],vectors,strict=True))
for pf in X.PROFILES:runtime.profile(pf)
retrievals=[];search_cache={}
for repeat in [1,2,3]:
 for q in queries:
  record=by_query[q['query_id'],repeat];route=record['routing'];original=record['original_scope']
  identity=f"{q['query_id']}:S_ONLY:r{repeat}";target=PATH/'retrieval'/(identity+'.json');new=not target.exists()
  if not new:row=P.F.read_json(target)
  else:
   result=M.rank_units(runtime,q['question'],M.structured_units(route))
   if original['clarify']:result['clarify']=True
   row=dict(identity=identity,query_id=q['query_id'],profile='common',mode='S_ONLY',repeat=repeat,result=result,route=route,original_scope=original,manifest_sha256=P.sha(manifest),binding_extra_inputs=None)
  M.bind_retrieval(54,row,question=q,profile='common',mode='S_ONLY',repeat=repeat,parser_record=record,manifest=manifest,new=new)
  if new:P.F.write_json(target,row)
  retrievals.append(row)
  for pf in X.PROFILES:
   results={}
   for mode in ['U','SEQ']:
    identity=f"{q['query_id']}:{pf}:{mode}:r{repeat}";target=PATH/'retrieval'/(identity+'.json');new=not target.exists()
    scope=original['card_keys'] if mode=='U' else route['scope']
    extra=dict(source53_selection=selected_AB[pf],selection_sha256=P.sha(selection)) if mode=='SEQ' else None
    key=P.sha(dict(question=q['question'],profile=pf,scope=scope))
    if not new:row=P.F.read_json(target)
    else:
     reuse=None;start=time.perf_counter()
     if original['clarify']:result=dict(cards=[],evidence=[],trace={},clarify=True)
     elif key in search_cache:result=copy.deepcopy(search_cache[key]);reuse='same question/profile/scope; original first-write search time, not new online time'
     else:result=runtime.search(q['question'],qvectors[q['query_id']],pf,scope)
     # The cache always holds the unchanged U body, before any answer-only supplement.
     search_cache[key]=copy.deepcopy(result)
     if mode=='SEQ' and selected_AB[pf]=='B':result=M.supplement(q['question'],result,route)
     row=dict(identity=identity,query_id=q['query_id'],profile=pf,mode=mode,repeat=repeat,result=result,route=None if mode=='U' else route,
      original_scope=original,manifest_sha256=P.sha(manifest),binding_extra_inputs=extra,reuse=reuse,observed_retrieval_wall_seconds=time.perf_counter()-start)
    M.bind_retrieval(54,row,question=q,profile=pf,mode=mode,repeat=repeat,parser_record=record,manifest=manifest,extra_inputs=extra,new=new)
    if new:P.F.write_json(target,row)
    if mode=='U':search_cache[key]=copy.deepcopy(row['result'])
    # Loaded SEQ+B must not contaminate a U cache; reconstruct its original evidence prefix.
    elif key not in search_cache:
     plain=copy.deepcopy(row['result'])
     if selected_AB[pf]=='B':
      plain['evidence']=[e for e in plain['evidence'] if e.get('origin')!='structured'];plain.pop('supplement',None)
     search_cache[key]=plain
    results[mode]=row['result'];retrievals.append(row)
   identity=f"{q['query_id']}:{pf}:PAR:r{repeat}";target=PATH/'retrieval'/(identity+'.json');new=not target.exists()
   if not new:row=P.F.read_json(target)
   else:
    units=[] if original['clarify'] else M.union_units(M.structured_units(route),M.unstructured_units(runtime,results['U'],pf))
    result=M.rank_units(runtime,q['question'],units)
    if original['clarify']:result['clarify']=True
    row=dict(identity=identity,query_id=q['query_id'],profile=pf,mode='PAR',repeat=repeat,result=result,route=route,original_scope=original,manifest_sha256=P.sha(manifest),binding_extra_inputs=None)
   M.bind_retrieval(54,row,question=q,profile=pf,mode='PAR',repeat=repeat,parser_record=record,manifest=manifest,independent_u=results['U'],new=new)
   if new:P.F.write_json(target,row)
   retrievals.append(row)
  P.F.write_json(PATH/'retrieval_progress.json',dict(logical_conditions=len(retrievals),expected=672,last_query=q['query_id'],repeat=repeat,ledger=X.ledger()))
  print('검색',q['query_id'],'회차',repeat,len(retrievals),flush=True)
retrieval_path=PATH/'retrieval_results.json'
if retrieval_path.exists():assert P.F.read_json(retrieval_path)['records']==retrievals
else:P.F.write_json(retrieval_path,dict(records=retrievals,resources=runtime.resources(),embedding_wall_seconds=runtime.last_embedding_seconds))


In [ ]:
require_reviewed_query_diff()
answers=[]
existing={p.stem for p in (PATH/'answers').glob('*.json')};initial_api={p.stem for p in (PATH/'api_cache').glob('*.json')}
ordered=sorted(retrievals,key=lambda r:(r['query_id'],r['repeat'],r['mode'] if (int(r['query_id'][1:])+r['repeat'])%2 else ''.join(chr(255-ord(c)) for c in r['mode']),r['profile']))
for row in ordered:
 q=next(q for q in queries if q['query_id']==row['query_id']);record=by_query[q['query_id'],row['repeat']]
 u=next((r['result'] for r in retrievals if r['query_id']==row['query_id'] and r['profile']==row['profile'] and r['mode']=='U' and r['repeat']==row['repeat']),None)
 M.bind_retrieval(54,row,question=q,profile=row['profile'],mode=row['mode'],repeat=row['repeat'],parser_record=record,manifest=manifest,independent_u=u if row['mode']=='PAR' else None,extra_inputs=row['binding_extra_inputs'])
 a=M.answer(54,row['identity'],q['question'],row['result'],allow_network=ALLOW_NETWORK)
 answers.append({**a,**{k:row[k] for k in ['query_id','profile','mode','repeat']}})
 P.F.write_json(PATH/'answer_progress.json',dict(completed=len(answers),expected=672,last_identity=row['identity'],ledger=X.ledger()))
 print('답변',row['identity'],a['status'],round(a['answer_ready_wall_seconds'],3),'초',flush=True)
segment=dict(resources=runtime.resources(),wall_seconds=time.perf_counter()-started,cpu_seconds=time.process_time()-cpu_started,
 newly_created_answer_count=len({a['identity'] for a in answers}-existing),new_physical_requests=len({p.stem for p in (PATH/'api_cache').glob('*.json')}-initial_api),time_scope='current segment; no online end-to-end claim')
completed=dict(answers=answers,retrievals=retrievals,manifest_sha256=P.sha(manifest),first_completed_segment=segment)
result_path=PATH/'execution_results.json'
if result_path.exists():
 saved=P.F.read_json(result_path);assert saved['answers']==answers and saved['retrievals']==retrievals and saved['manifest_sha256']==P.sha(manifest)
else:P.F.write_json(result_path,completed)
P.F.write_json(PATH/'latest_completed_segment.json',segment)
P.F.write_json(PATH/'completion_status.json',dict(status='awaiting_source_evaluation',logical_conditions=len(answers),expected=672,operating_changes=False))
print('3회 전체 실행 완료. 원문 평가 전 운영 후보 확정 없음.')


## 원문 평가와 선정
전체672조건을 원문으로 판정해 source_judgments.json에 답변·질문·출처 해시와 PASS/PARTIAL/FAIL/UNCERTAIN을 저장한다. 원문 근거가 부족한 경우는 통과로 세지 않는다. 카드별 실제 조건으로 판정하며 다른 카드의 수치를 대입하지 않는다. 기준U와 비교한 새 중요 오류 및 조건 손실은 오류 유무만 비교하지 않고 개별 내용으로 확인한다. 모든 하위집단을 함께 공개한다. 선정 조건과 비용·시간을 정리한 뒤 HTML을 직접 작성한다. 이 셀의 설명만으로 평가 완료를 선언하지 않는다.

In [ ]:
# 평가 판정과 원문 확인 자료는 답변 생성 LLM에 보내지 않는다.
# 질문·답변·PDF 소유 관계를 확인한 Codex 판정을 전체 분모로 집계한다.
judgment_path=PATH/'source_judgments.json'
if not judgment_path.exists():
 print('672조건 원문 판정 대기. 최종 후보 미정.')
else:
 require_reviewed_query_diff()
 completed=P.F.read_json(PATH/'execution_results.json')
 assert completed['manifest_sha256']==P.sha(manifest)
 for name,wanted in manifest['source_input_sha256'].items():assert P.sha((ROOT/name).read_bytes())==wanted
 assert P.sha(Q.documents())==manifest['corpus_documents_sha256']
 rows=completed['answers'];retrieval_by_id={r['identity']:r for r in completed['retrievals']}
 annotations=P.F.read_json(judgment_path);judged={r['identity']:r for r in annotations['records']}
 expected=set(manifest['allowed_identities']['answer'])
 assert len(rows)==len(judged)==len(annotations['records'])==672 and set(judged)=={r['identity'] for r in rows}==expected
 for row in rows:
  q=next(q for q in queries if q['query_id']==row['query_id']);j=judged[row['identity']];retrieval=retrieval_by_id[row['identity']]
  assert j['answer_sha256']==P.sha(row) and j['question_sha256']==P.sha(q['question']) and j['reason']
  E.validate_judgment(j,rubric_by_query[row['query_id']],P.sha)
  assert type(j['important_error']) is bool and type(j['scope_error']) is bool
  profiles=X.PROFILES if row['mode']=='S_ONLY' else [row['profile']]
  for field in ['new_important_error_vs_U','condition_loss_vs_U']:
   assert set(j[field])==set(profiles) and all(type(v) is bool for v in j[field].values())
   if row['mode']=='U':assert not any(j[field].values())
  if any(j['new_important_error_vs_U'].values()):assert j['important_error']
  assert (PATH/'answers'/(row['identity']+'.json')).exists()
  raw=M.answer(54,row['identity'],q['question'],retrieval['result'],allow_network=False)
  assert row=={**raw,**{k:row[k] for k in ['query_id','profile','mode','repeat']}}
  if row['status']!='ok':assert j['verdict']!='PASS'
  retrieved={c['card_key'] for c in retrieval['result']['cards']+retrieval['result']['evidence']}
  assert j['source_support']
  for support in j['source_support']:
   if support.get('kind')=='corpus_inventory':
    assert not gold_by_query[q['query_id']] and q['subgroup']=='negative'
    assert support['documents_sha256']==manifest['corpus_documents_sha256'] and support['observation']
    continue
   key=support['card_key'];name='data/raw/'+key+'.pdf'
   assert support['path']==name and support['sha256']==manifest['source_input_sha256'][name]
   assert type(support['page']) is int and 1<=support['page']<=manifest['source_page_counts'][key] and support['observation']
   assert support['relation'] in ['retrieved_card','missing_expected_card']
   if support['relation']=='retrieved_card':assert key in retrieved
   else:assert j['verdict']!='PASS' and key in gold_by_query[q['query_id']]
 lookup={(r['query_id'],r['profile'],r['mode'],r['repeat']):judged[r['identity']] for r in rows}
 summaries=[]
 for pf in X.PROFILES:
  for mode in ['S_ONLY','U','SEQ','PAR']:
   mpf='common' if mode=='S_ONLY' else pf
   pairs=[];new_errors=[];condition_losses=[];scope_errors=[];subgroups={}
   for q in queries:
    b=[lookup[q['query_id'],pf,'U',r] for r in [1,2,3]]
    a=[lookup[q['query_id'],mpf,mode,r] for r in [1,2,3]]
    base=sum(j['verdict']=='PASS' for j in b);candidate=sum(j['verdict']=='PASS' for j in a)
    pairs.append(dict(query_id=q['query_id'],subgroup=q['subgroup'],U_passes=base,candidate_passes=candidate,outcome='win' if candidate>base else 'loss' if candidate<base else 'tie'))
    group=subgroups.setdefault(q['subgroup'],dict(questions=0,logical_conditions=0,U_passes=0,candidate_passes=0))
    group['questions']+=1;group['logical_conditions']+=3;group['U_passes']+=base;group['candidate_passes']+=candidate
    for j in a:
     if j['new_important_error_vs_U'][pf]:new_errors.append(j['identity'])
     if j['condition_loss_vs_U'][pf]:condition_losses.append(j['identity'])
     if j['scope_error']:scope_errors.append(j['identity'])
   wlt={k:sum(p['outcome']==k for p in pairs) for k in ['win','loss','tie']}
   declines=[g for g,s in subgroups.items() if s['candidate_passes']<s['U_passes']]
   eligible=mode!='U' and wlt['win']-wlt['loss']>=1 and not (declines or new_errors or condition_losses or scope_errors)
   repeat_passes=[sum(lookup[q['query_id'],mpf,mode,r]['verdict']=='PASS' for q in queries) for r in [1,2,3]]
   from statistics import pstdev
   summaries.append(dict(repeat_passes=repeat_passes,population_std_of_repeat_pass_rates=pstdev([v/32 for v in repeat_passes]),std_denominator=3,profile=pf,mode=mode,question_denominator=32,repeats=3,logical_denominator=96,PASS=sum(p['candidate_passes'] for p in pairs),
    question_wins_losses_ties=wlt,subgroups=subgroups,new_important_error_ids=new_errors,condition_loss_ids=condition_losses,scope_error_ids=scope_errors,
    subgroup_declines=declines,eligible_operating_candidate=eligible,question_pairs=pairs))
 summary=dict(primary_metric='question_verdict',secondary_metric='annotation_coverage',question_rubrics_sha256=P.sha(rubrics),split='confirmatory_same_corpus',execution_results_sha256=P.sha(completed),source_judgments_sha256=P.sha(annotations),
  unique_logical_conditions=672,common_S_ONLY_conditions=96,configuration_tables_repeat_common_S_ONLY=True,configurations=summaries,independent_human_review=False,production_adopted=False,rule=manifest['decision'])
 X.frozen(PATH/'evaluation_summary.json',summary)
 P.F.write_json(PATH/'completion_status.json',dict(status='completed_confirmatory_comparison',logical_conditions=672,expected=672,operating_changes=False,production_adopted=False))
 print(json.dumps([{k:v for k,v in r.items() if k not in ['question_pairs','subgroups']} for r in summaries],ensure_ascii=False,indent=2))
